In [ ]:
import os
import shutil
import random
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
TRAIN_DIR = Path('/kaggle/working/voice_train')
TEST_DIR = Path('/kaggle/working/voice_unseen_test')

for d in [TRAIN_DIR, TEST_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# Quét toàn bộ file .wav từ các bộ dữ liệu đã gắn (bỏ qua file nhiễu nền)
all_wavs = [p for p in INPUT_ROOT.rglob('*.wav') if 'noise' not in str(p).lower()]
assert len(all_wavs) > 0, "Chưa gắn bộ dữ liệu âm thanh nào vào Input!"

speakers = {}
for w in all_wavs:
    # Tạo mã người nói duy nhất theo tên dataset + thư mục cha
    ds_name = w.parts[3] if len(w.parts) > 3 else "ds"
    spk_id = f"{ds_name}_{w.parent.name}"
    speakers.setdefault(spk_id, []).append(w)

# Lọc các người nói có ít nhất 20 file ghi âm
valid_spks = sorted([k for k, v in speakers.items() if len(v) >= 20])
random.seed(42)
random.shuffle(valid_spks)

# Dành 85% người nói để train, 15% người nói chưa từng thấy để đo ngưỡng mở khóa
split_idx = max(1, int(len(valid_spks) * 0.85))
train_spks = valid_spks[:split_idx]
test_spks = valid_spks[split_idx:]

for spk in train_spks:
    dst = TRAIN_DIR / spk
    dst.mkdir(parents=True, exist_ok=True)
    for idx, w in enumerate(speakers[spk][:400]):
        os.symlink(w, dst / f"{idx:04d}_{w.name}")

for spk in test_spks:
    dst = TEST_DIR / spk
    dst.mkdir(parents=True, exist_ok=True)
    for idx, w in enumerate(speakers[spk][:50]):
        os.symlink(w, dst / f"{idx:04d}_{w.name}")

print(f"HOÀN TẤT! Train trên {len(train_spks)} người ({len(list(TRAIN_DIR.rglob('*.wav')))} file)")
print(f"Kiểm thử ngưỡng trên {len(test_spks)} người chưa từng gặp ({len(list(TEST_DIR.rglob('*.wav')))} file)")

In [ ]:
!pip install -q soundfile scipy scikit-learn
import random
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
from pathlib import Path

SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE * 3  # Chuẩn 3 giây = 48000 samples

mel_extractor = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=512, win_length=400, hop_length=160, n_mels=64
)

def load_audio_3s(path, augment=False):
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        audio = resample_poly(audio, SAMPLE_RATE, sr).astype(np.float32)

    # Cắt khoảng lặng đầu/cuối và chuẩn hóa biên độ
    peak = np.max(np.abs(audio))
    if peak > 1e-4:
        audio = audio / peak * 0.9

    # Nếu clip ngắn (như AudioMNIST ~1s), lặp lại tự nhiên để mô phỏng câu nói 3s
    if len(audio) < TARGET_LEN:
        repeats = int(np.ceil(TARGET_LEN / max(1, len(audio))))
        audio = np.tile(audio, repeats)[:TARGET_LEN]
    elif len(audio) > TARGET_LEN:
        start = random.randint(0, len(audio) - TARGET_LEN) if augment else (len(audio) - TARGET_LEN) // 2
        audio = audio[start:start + TARGET_LEN]

    if augment:
        # Giả lập khoảng cách đứng gần/xa mic và nhiễu trắng của mạch ESP32
        audio = audio * random.uniform(0.6, 1.2)
        if random.random() < 0.6:
            audio = audio + np.random.normal(0, random.uniform(0.002, 0.012), size=audio.shape).astype(np.float32)

    wav_t = torch.from_numpy(audio).unsqueeze(0)
    mel = torch.log(mel_extractor(wav_t) + 1e-6)  # (1, 64, 301)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    if augment:
        # Giả lập dải tần hẹp của mic MEMS ESP32 (cắt ngẫu nhiên dải trầm/cao) + SpecAugment
        if random.random() < 0.5:
            mel[:, :random.randint(2, 5), :] = 0.0
            mel[:, -random.randint(3, 8):, :] = 0.0
        if random.random() < 0.5:
            t0 = random.randint(0, 260)
            mel[:, :, t0:t0 + random.randint(5, 25)] = 0.0
    return mel

class VoiceDataset(Dataset):
    def __init__(self, root_dir, augment=True):
        self.samples = []
        self.classes = sorted([d.name for d in Path(root_dir).iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        for c in self.classes:
            for p in (Path(root_dir) / c).glob('*.wav'):
                self.samples.append((str(p), self.class_to_idx[c]))
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return load_audio_3s(path, self.augment), label

train_ds = VoiceDataset('/kaggle/working/voice_train', augment=True)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, in_c, out_c, k=3, dilation=1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_c, out_c, k, padding=pad, dilation=dilation, bias=False),
            nn.BatchNorm1d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_c, out_c, k, padding=pad, dilation=dilation, bias=False),
            nn.BatchNorm1d(out_c),
            SEBlock1D(out_c)
        )
        self.shortcut = nn.Conv1d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))

class SpeakerEmbedder(nn.Module):
    def __init__(self, emb_size=256):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv1d(64, 128, 5, padding=2, bias=False), nn.BatchNorm1d(128), nn.ReLU())
        self.layer2 = ResBlock1D(128, 128, k=3, dilation=2)
        self.layer3 = ResBlock1D(128, 256, k=3, dilation=3)
        self.layer4 = ResBlock1D(256, 256, k=3, dilation=4)
        self.attn = nn.Sequential(nn.Conv1d(256, 64, 1), nn.Tanh(), nn.Conv1d(64, 256, 1), nn.Softmax(dim=-1))
        self.fc = nn.Sequential(nn.Linear(512, emb_size, bias=False), nn.BatchNorm1d(emb_size))

    def forward(self, x):
        x = x.squeeze(1)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        w = self.attn(x)
        mean = torch.sum(x * w, dim=-1)
        std = torch.sqrt(torch.clamp(torch.sum(((x - mean.unsqueeze(-1)) ** 2) * w, dim=-1), min=1e-6))
        return F.normalize(self.fc(torch.cat([mean, std], dim=1)), p=2, dim=1)

class ArcFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.40):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s, self.m = s, m

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-6, 1.0 - 1e-6))
        target_logits = torch.cos(theta + self.m)
        one_hot = F.one_hot(labels, num_classes=self.weight.size(0)).float()
        return (one_hot * target_logits + (1.0 - one_hot) * cosine) * self.s

device = torch.device('cuda')
spk_model = SpeakerEmbedder(emb_size=256).to(device)
arcface = ArcFaceHead(in_features=256, out_features=len(train_ds.classes)).to(device)

optimizer = torch.optim.AdamW(list(spk_model.parameters()) + list(arcface.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(1, 21):
    spk_model.train()
    arcface.train()
    total_loss, correct, total = 0.0, 0, 0
    for mels, labels in train_loader:
        mels, labels = mels.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            emb = spk_model(mels)
            logits = arcface(emb, labels)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * mels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += mels.size(0)
    scheduler.step()
    print(f"Epoch {epoch:02d}/20 | Loss: {total_loss/total:.4f} | Acc: {correct/total*100:.2f}%")

torch.save(spk_model.state_dict(), '/kaggle/working/speaker_embedder.pth')

In [ ]:
!pip install -q onnx onnxscript

spk_model.eval()

# 1. Xuất model trích xuất vector giọng nói 256 chiều sang ONNX
dummy = torch.randn(1, 1, 64, 301, device=device)
torch.onnx.export(
    spk_model, dummy, '/kaggle/working/speaker_arcface.onnx',
    input_names=['mel_input'], output_names=['embedding'],
    dynamic_axes={'mel_input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=12,
    dynamo=False
)

@torch.no_grad()
def get_emb(wav_path):
    mel = load_audio_3s(wav_path, augment=False).unsqueeze(0).to(device)
    return spk_model(mel).cpu().numpy()[0]

# 2. Kiểm tra khả năng phân biệt trên nhóm người nói hoàn toàn mới (chưa từng train)
test_embs = {}
for spk_dir in sorted(Path('/kaggle/working/voice_unseen_test').glob('*')):
    wavs = sorted(spk_dir.glob('*.wav'))[:20]
    if len(wavs) >= 5:
        test_embs[spk_dir.name] = [get_emb(w) for w in wavs]

same_scores, diff_scores = [], []
spk_names = list(test_embs.keys())
for i, spk_a in enumerate(spk_names):
    embs_a = test_embs[spk_a]
    for x in range(len(embs_a)):
        for y in range(x + 1, len(embs_a)):
            same_scores.append(float(embs_a[x] @ embs_a[y]))
    for spk_b in spk_names[i + 1:]:
        for ea in embs_a[:5]:
            for eb in test_embs[spk_b][:5]:
                diff_scores.append(float(ea @ eb))

print(f"Điểm tương đồng CÙNG 1 NGƯỜI (chưa từng train): {np.mean(same_scores):.4f}")
print(f"Điểm tương đồng KHÁC NGƯỜI (người lạ):         {np.mean(diff_scores):.4f}")
print(f"-> Ngưỡng xác thực giọng nói đề xuất: {(np.mean(same_scores) + np.mean(diff_scores))/2:.2f}")